# GPT

O código é adaptado [GPT tutorial](https://keras.io/examples/generative/text_generation_with_miniature_gpt/) criado por Apoorv Nandan.

Ref.: David Foster. Generative Deep Learning. O'Reilly Media; 2nd ed. 2023.

Link para o original: https://github.com/davidADSP/Generative_Deep_Learning_2nd_Edition/blob/main/notebooks/09_transformer/gpt/gpt.ipynb

In [ ]:
import numpy as np
import json
import re
import string
from IPython.display import display, HTML

import tensorflow as tf
from tensorflow.keras import layers, models, losses, callbacks

## 0. Parameters <a name="parameters"></a>

In [ ]:
VOCAB_SIZE = 10000
MAX_LEN = 80
EMBEDDING_DIM = 256
KEY_DIM = 256
N_HEADS = 2
FEED_FORWARD_DIM = 256
VALIDATION_SPLIT = 0.2
SEED = 42
LOAD_MODEL = False
BATCH_SIZE = 32
EPOCHS = 2

## 1. Load the data <a name="load"></a>

In [ ]:
# Load the full dataset
with open("winemag-data-130k-v2.json") as json_data:
    wine_data = json.load(json_data)

In [ ]:
len(wine_data)

129971

In [ ]:
wine_data[10]

{'points': '87',
 'title': 'Kirkland Signature 2011 Mountain Cuvée Cabernet Sauvignon (Napa Valley)',
 'description': 'Soft, supple plum envelopes an oaky structure in this Cabernet, supported by 15% Merlot. Coffee and chocolate complete the picture, finishing strong at the end, resulting in a value-priced wine of attractive flavor and immediate accessibility.',
 'taster_name': 'Virginie Boone',
 'taster_twitter_handle': '@vboone',
 'price': 19,
 'designation': 'Mountain Cuvée',
 'variety': 'Cabernet Sauvignon',
 'region_1': 'Napa Valley',
 'region_2': 'Napa',
 'province': 'California',
 'country': 'US',
 'winery': 'Kirkland Signature'}

In [ ]:
# Filter the dataset
filtered_data = [
    "wine review : "
    + x["country"]
    + " : "
    + x["province"]
    + " : "
    + x["variety"]
    + " : "
    + x["description"]
    for x in wine_data
    if x["country"] is not None
    and x["province"] is not None
    and x["variety"] is not None
    and x["description"] is not None
]

In [ ]:
# Count the recipes
n_wines = len(filtered_data)
print(f"{n_wines} recipes loaded")

129907 recipes loaded


In [ ]:
example = filtered_data[25]
print(example)

wine review : US : California : Pinot Noir : Oak and earth intermingle around robust aromas of wet forest floor in this vineyard-designated Pinot that hails from a high-elevation site. Small in production, it offers intense, full-bodied raspberry and blackberry steeped in smoky spice and smooth texture.



## 2. Tokenize the data <a name="tokenize"></a>

In [ ]:
# Pad the punctuation, to treat them as separate 'words'
def pad_punctuation(s):
    s = re.sub(f"([{string.punctuation}, '\n'])", r" \1 ", s)
    s = re.sub(" +", " ", s)
    return s


text_data = [pad_punctuation(x) for x in filtered_data]

In [ ]:
# Display an example of a recipe
example_data = text_data[25]
example_data

'wine review : US : California : Pinot Noir : Oak and earth intermingle around robust aromas of wet forest floor in this vineyard - designated Pinot that hails from a high - elevation site . Small in production , it offers intense , full - bodied raspberry and blackberry steeped in smoky spice and smooth texture . '

In [ ]:
# Convert to a Tensorflow Dataset
text_ds = (
    tf.data.Dataset.from_tensor_slices(text_data)
    .batch(BATCH_SIZE)
    .shuffle(1000)
)

In [ ]:
# Create a vectorisation layer
vectorize_layer = layers.TextVectorization(
    standardize="lower",
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_LEN + 1,
)

In [ ]:
# Adapt the layer to the training set
vectorize_layer.adapt(text_ds)
vocab = vectorize_layer.get_vocabulary()

In [ ]:
# Display some token:word mappings
for i, word in enumerate(vocab[:10]):
    print(f"{i}: {word}")

0: 
1: [UNK]
2: :
3: ,
4: .
5: and
6: the
7: wine
8: a
9: of


In [ ]:
# Display the same example converted to ints
example_tokenised = vectorize_layer(example_data)
print(example_tokenised.numpy())

[   7   10    2   20    2   29    2   43   62    2   55    5  243 4145
  453  634   26    9  497  499  667   17   12  142   14 2214   43   25
 2484   32    8  223   14 2213  948    4  594   17  987    3   15   75
  237    3   64   14   82   97    5   74 2633   17  198   49    5  125
   77    4    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0]


## 3. Create the Training Set <a name="create"></a>

In [ ]:
# Create the training set of recipes and the same text shifted by one word
def prepare_inputs(text):
    text = tf.expand_dims(text, -1)
    tokenized_sentences = vectorize_layer(text)
    x = tokenized_sentences[:, :-1]
    y = tokenized_sentences[:, 1:]
    return x, y


train_ds = text_ds.map(prepare_inputs)

In [ ]:
example_input_output = train_ds.take(1).get_single_element()

In [ ]:
# Example Input
example_input_output[0][0]

<tf.Tensor: shape=(80,), dtype=int64, numpy=
array([   7,   10,    2,   42,    2,  255,   98,    2,  255,   14,   63,
         24,   27,    2,    1,   18,   21,  320,   39,  222,  852,  575,
          4,   12,  161,    3,    8,   27,    9,  641,   48,  286,    3,
        969,   48,   92,    3,  660,   48,  563,    5,  459,   48, 1454,
          3,  673, 1172,   94,  489,    5, 1166,   26,  167,   36,   22,
          4,   15,   18,   21,  303,    5,  184,   23,    6,   28,    3,
         11,    8,  134,   31,    4,   35, 2701,    4,    0,    0,    0,
          0,    0,    0])>

In [ ]:
# Example Output (shifted by one token)
example_input_output[1][0]

<tf.Tensor: shape=(80,), dtype=int64, numpy=
array([  10,    2,   42,    2,  255,   98,    2,  255,   14,   63,   24,
         27,    2,    1,   18,   21,  320,   39,  222,  852,  575,    4,
         12,  161,    3,    8,   27,    9,  641,   48,  286,    3,  969,
         48,   92,    3,  660,   48,  563,    5,  459,   48, 1454,    3,
        673, 1172,   94,  489,    5, 1166,   26,  167,   36,   22,    4,
         15,   18,   21,  303,    5,  184,   23,    6,   28,    3,   11,
          8,  134,   31,    4,   35, 2701,    4,    0,    0,    0,    0,
          0,    0,    0])>

## 5. Create the causal attention mask function <a name="causal"></a>

In [ ]:
def causal_attention_mask(batch_size, n_dest, n_src, dtype):
    i = tf.range(n_dest)[:, None]
    j = tf.range(n_src)
    m = i >= j - n_src + n_dest
    mask = tf.cast(m, dtype)
    mask = tf.reshape(mask, [1, n_dest, n_src])
    mult = tf.concat(
        [tf.expand_dims(batch_size, -1), tf.constant([1, 1], dtype=tf.int32)], 0
    )
    return tf.tile(mask, mult)


np.transpose(causal_attention_mask(1, 10, 10, dtype=tf.int32)[0])

array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]], dtype=int32)

## 6. Create a Transformer Block layer <a name="transformer"></a>

In [ ]:
class TransformerBlock(layers.Layer):
    def __init__(self, num_heads, key_dim, embed_dim, ff_dim, dropout_rate=0.1):
        super(TransformerBlock, self).__init__()
        self.num_heads = num_heads
        self.key_dim = key_dim
        self.embed_dim = embed_dim
        self.ff_dim = ff_dim
        self.dropout_rate = dropout_rate
        self.attn = layers.MultiHeadAttention(
            num_heads, key_dim, output_shape=embed_dim
        )
        self.dropout_1 = layers.Dropout(self.dropout_rate)
        self.ln_1 = layers.LayerNormalization(epsilon=1e-6)
        self.ffn_1 = layers.Dense(self.ff_dim, activation="relu")
        self.ffn_2 = layers.Dense(self.embed_dim)
        self.dropout_2 = layers.Dropout(self.dropout_rate)
        self.ln_2 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size = input_shape[0]
        seq_len = input_shape[1]
        causal_mask = causal_attention_mask(
            batch_size, seq_len, seq_len, tf.bool
        )
        attention_output, attention_scores = self.attn(
            inputs,
            inputs,
            attention_mask=causal_mask,
            return_attention_scores=True,
        )
        attention_output = self.dropout_1(attention_output)
        out1 = self.ln_1(inputs + attention_output)
        ffn_1 = self.ffn_1(out1)
        ffn_2 = self.ffn_2(ffn_1)
        ffn_output = self.dropout_2(ffn_2)
        return (self.ln_2(out1 + ffn_output), attention_scores)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "key_dim": self.key_dim,
                "embed_dim": self.embed_dim,
                "num_heads": self.num_heads,
                "ff_dim": self.ff_dim,
                "dropout_rate": self.dropout_rate,
            }
        )
        return config

## 7. Create the Token and Position Embedding <a name="embedder"></a>

In [ ]:
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, max_len, vocab_size, embed_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        self.max_len = max_len
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.token_emb = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim
        )
        self.pos_emb = layers.Embedding(input_dim=max_len, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "max_len": self.max_len,
                "vocab_size": self.vocab_size,
                "embed_dim": self.embed_dim,
            }
        )
        return config

## 8. Build the Transformer model <a name="transformer_decoder"></a>

In [ ]:
inputs = layers.Input(shape=(None,), dtype=tf.int32)
x = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBEDDING_DIM)(inputs)
x, attention_scores = TransformerBlock(
    N_HEADS, KEY_DIM, EMBEDDING_DIM, FEED_FORWARD_DIM
)(x)
outputs = layers.Dense(VOCAB_SIZE, activation="softmax")(x)
gpt = models.Model(inputs=inputs, outputs=[outputs, attention_scores])
gpt.compile("adam", loss=[losses.SparseCategoricalCrossentropy(), None])

In [ ]:
gpt.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, None, 256)      │     2,580,480 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ [(None, None, 256),    │       658,688 │
│ (TransformerBlock)              │ (None, 2, None, None)] │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, None, 10000)    │     2,570,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,809,168 (22.16 MB)

 Trainable params: 5,809,168 (22.16 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
if LOAD_MODEL:
    # model.load_weights('./models/model')
    gpt = models.load_model("./models/gpt", compile=True)

## 9. Train the Transformer <a name="train"></a>

In [ ]:
# Create a TextGenerator checkpoint
class TextGenerator(callbacks.Callback):
    def __init__(self, index_to_word, top_k=10):
        self.index_to_word = index_to_word
        self.word_to_index = {
            word: index for index, word in enumerate(index_to_word)
        }

    def sample_from(self, probs, temperature):
        probs = probs ** (1 / temperature)
        probs = probs / np.sum(probs)
        return np.random.choice(len(probs), p=probs), probs

    def generate(self, start_prompt, max_tokens, temperature):
        start_tokens = [
            self.word_to_index.get(x, 1) for x in start_prompt.split()
        ]
        sample_token = None
        info = []
        while len(start_tokens) < max_tokens and sample_token != 0:
            x = np.array([start_tokens])
            y, att = self.model.predict(x, verbose=0)
            sample_token, probs = self.sample_from(y[0][-1], temperature)
            info.append(
                {
                    "prompt": start_prompt,
                    "word_probs": probs,
                    "atts": att[0, :, -1, :],
                }
            )
            start_tokens.append(sample_token)
            start_prompt = start_prompt + " " + self.index_to_word[sample_token]
        print(f"\ngenerated text:\n{start_prompt}\n")
        return info

    def on_epoch_end(self, epoch, logs=None):
        self.generate("wine review", max_tokens=80, temperature=1.0)

In [ ]:
# Create a model save checkpoint
model_checkpoint_callback = callbacks.ModelCheckpoint(
    filepath="./checkpoint/checkpoint.weights.h5",
    save_weights_only=True,
    save_freq="epoch",
    verbose=0,
)

tensorboard_callback = callbacks.TensorBoard(log_dir="./logs")

# Tokenize starting prompt
text_generator = TextGenerator(vocab)

In [ ]:
gpt.fit(
    train_ds,
    epochs=EPOCHS,
    callbacks=[model_checkpoint_callback, tensorboard_callback, text_generator],
)

Epoch 1/2
4060/4060 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 2.5950
generated text:
wine review : us : california : riesling : dill , apricot , apricot , meyer lemon and honey flavors combine for a crisp , softly fruity riesling . there is an acidic kick in this wine , a fine wine whose effervescence alcohol identified - inspired citrus notes and a slight kamptal . 

4060/4060 ━━━━━━━━━━━━━━━━━━━━ 199s 46ms/step - loss: 2.2444
Epoch 2/2
4059/4060 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 1.9788
generated text:
wine review : italy : sicily & sardinia : red blend : underbrush scents of pressed violet and perfumed berry and blue flower float out of the glass . on the palate are already evolved with lithe flavors of black cherry , raisin , clove and chopped herb alongside velvety tannins . 

4060/4060 ━━━━━━━━━━━━━━━━━━━━ 131s 32ms/step - loss: 1.9573


In [ ]:
# Save the final model
gpt.save("gpt.keras")

# 3. Generate text using the Transformer

In [ ]:
def print_probs(info, vocab, top_k=5):
    for i in info:
        highlighted_text = []
        for word, att_score in zip(
            i["prompt"].split(), np.mean(i["atts"], axis=0)
        ):
            highlighted_text.append(
                '<span style="background-color:rgba(135,206,250,'
                + str(att_score / max(np.mean(i["atts"], axis=0)))
                + ');">'
                + word
                + "</span>"
            )
        highlighted_text = " ".join(highlighted_text)
        display(HTML(highlighted_text))

        word_probs = i["word_probs"]
        p_sorted = np.sort(word_probs)[::-1][:top_k]
        i_sorted = np.argsort(word_probs)[::-1][:top_k]
        for p, i in zip(p_sorted, i_sorted):
            print(f"{vocab[i]}:   \t{np.round(100*p,2)}%")
        print("--------\n")

**Gerando texto**

Podemos gerar um novo texto aplicando o seguinte processo:

* Alimente a rede com uma sequência de palavras existente e peça para prever a palavra seguinte.

* Anexe esta palavra à sequência existente e repita.

A rede produzirá um conjunto de probabilidades para cada palavra da qual podemos fazer uma amostra

In [ ]:
info = text_generator.generate(
    "wine review : us", max_tokens=80, temperature=1.0
)


generated text:
wine review : us : virginia : chardonnay : this wine is light and elegant , well - made and flavorful with ample aeration from chile ' s top separate vineyards in the pool . toasted [UNK] present complex aromas of pineapple , lemon spritzer , custard peel and butter . the wine feels starting with ripe vanilla and cream flavors , with a velvety texture that lingers on a textured finish . 



In [ ]:
info = text_generator.generate(
    "wine review : italy", max_tokens=80, temperature=0.5
)


generated text:
wine review : italy : piedmont : nebbiolo : this opens with aromas of rose , violet , underbrush , baked plum and a whiff of game . the palate delivers mature black cherry , licorice , ground pepper and a hint of clove alongside youthfully assertive tannins that need time to unwind . drink after 2018 . 



Ambos são semelhantes a uma crítica de vinhos do conjunto de treinamento original.

Ambos abrem com a região e o tipo de vinho, e o tipo de vinho permanece consistente ao longo da passagem (por exemplo, não muda de cor no meio).

O texto gerado com temperatura 1,0 é mais aventureiro e, portanto, menos preciso do que o exemplo com temperatura 0,5.

A geração de múltiplas amostras com temperatura 1,0 levará, portanto, a mais variedade, pois o modelo amostra uma distribuição de probabilidade com maior variância.

In [ ]:
info = text_generator.generate(
    "wine review : germany", max_tokens=80, temperature=0.5
)
print_probs(info, vocab)


generated text:
wine review : germany : mosel : riesling : whiffs of struck flint and petrol are accented by a hint of mineral . this intensely aromatic wine is dry in style , with a crystalline edge of grapefruit and bramble . it ' s a refreshingly light - footed wine that ' s refreshingly spry and spry . 



::   	100.0%
-:   	0.0%
grosso:   	0.0%
zealand:   	0.0%
giallo:   	0.0%
--------



mosel:   	87.58999633789062%
pfalz:   	7.039999961853027%
rheinhessen:   	2.859999895095825%
nahe:   	1.1399999856948853%
rheingau:   	1.0%
--------



::   	99.88999938964844%
-:   	0.10999999940395355%
hills:   	0.0%
valley:   	0.0%
grosso:   	0.0%
--------



riesling:   	99.97000122070312%
gewürztraminer:   	0.019999999552965164%
pinot:   	0.0%
sparkling:   	0.0%
white:   	0.0%
--------



...

brisk:   	34.83000183105469%
quaffable:   	30.43000030517578%
crisp:   	13.680000305175781%
spry:   	10.069999694824219%
refreshing:   	4.929999828338623%
--------



and:   	40.119998931884766%
,:   	22.6200008392334%
on:   	16.639999389648438%
with:   	9.859999656677246%
.:   	6.610000133514404%
--------



spry:   	85.69000244140625%
refreshing:   	5.480000019073486%
revitalizing:   	1.0399999618530273%
penetrating:   	0.8999999761581421%
zesty:   	0.7599999904632568%
--------



.:   	96.62999725341797%
,:   	2.930000066757202%
on:   	0.2800000011920929%
with:   	0.07999999821186066%
in:   	0.019999999552965164%
--------



:   	53.29999923706055%
drink:   	39.63999938964844%
enjoy:   	2.5899999141693115%
it:   	2.0399999618530273%
finishes:   	1.340000033378601%
--------



In [ ]:
info = text_generator.generate(
    "wine review : us", max_tokens=80, temperature=0.5
)
print_probs(info, vocab)


generated text:
wine review : us : california : zinfandel : this is a big , delicious , rich and full - bodied , with smooth tannins and a velvety texture . it smells like black cherry , black tea and black pepper . it ' s a bit brusque now , with a touch of astringency that suggests pairing with lamb , beef char , roasted lamb . 



::   	100.0%
-:   	0.0%
grosso:   	0.0%
africa:   	0.0%
[UNK]:   	0.0%
--------



california:   	96.02999877929688%
washington:   	2.240000009536743%
oregon:   	1.5199999809265137%
new:   	0.10999999940395355%
virginia:   	0.07999999821186066%
--------



::   	100.0%
grosso:   	0.0%
[UNK]:   	0.0%
-:   	0.0%
province:   	0.0%
--------



cabernet:   	27.219999313354492%
chardonnay:   	16.3700008392334%
pinot:   	14.850000381469727%
merlot:   	10.729999542236328%
sauvignon:   	9.579999923706055%
--------



...

and:   	66.69999694824219%
,:   	31.700000762939453%
or:   	0.7400000095367432%
.:   	0.7099999785423279%
-:   	0.11999999731779099%
--------



roasted:   	36.4900016784668%
beef:   	19.8799991607666%
lamb:   	15.550000190734863%
roast:   	4.579999923706055%
black:   	3.740000009536743%
--------



meat:   	48.66999816894531%
lamb:   	23.200000762939453%
meats:   	8.5600004196167%
beef:   	8.15999984741211%
game:   	6.760000228881836%
--------



.:   	36.02000045776367%
,:   	27.299999237060547%
and:   	24.049999237060547%
or:   	6.610000133514404%
chops:   	4.840000152587891%
--------



:   	99.8499984741211%
drink:   	0.10999999940395355%
it:   	0.019999999552965164%
the:   	0.009999999776482582%
try:   	0.0%
--------



Ref.: David Foster. Generative Deep Learning. O'Reilly Media; 2nd ed. 2023.

Link para o original: https://github.com/davidADSP/Generative_Deep_Learning_2nd_Edition/blob/main/notebooks/09_transformer/gpt/gpt.ipynb